# LendingClub 전처리 근거 감사 리포트 (Section 3: 표본 · MIRR · 변수 선별)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Walnutbrown/kdt14th-lendingclub/blob/main/notebooks/01b_eda_audit.ipynb)

> **역할 분리 원칙 (SRP):**
> - 재사용 전처리·국채매칭·`MIRR` 함수는 **`lib_prep.ipynb`**에 있습니다 (2026-11-24 `preprocess.py`에서 옮김).
> - 본 노트북은 논문 Section 3에 수록할 **3가지 실증 근거 표**를 1회 실행으로 재현하기 위한 **감사 리포트 전용**입니다:
>   1. **표 1:** 만기별·발행월별 미종결률 실측 (우측 중도절단 컷오프 `issue_d + term ≤ 2020-04` 근거)
>   2. **표 2:** 만기 대응 국채(`DGS3`/`DGS5`) 매칭 및 현금흐름 복원 `MIRR`·초과수익률($R_i^{\text{excess}}$) 분포
>   3. **표 3:** 타깃($y$) 미사용 6단계 변수 선별 깔때기 (`141 → 37` 원천, `|r| ≤ 0.75`, `VIF < 5.0`)


> 2026-11-25: 01_eda_audit(11-24)에서 불러오는 경로만 바꾸고, 표를 `report/eda_audit/`에 저장하도록 했습니다. 4장(전처리기 정합성)은 `01_prep`이 같은 일을 하므로 뺐습니다.

In [1]:
import sys
if "google.colab" in sys.modules:
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb

DEV_CSV = TRAIN_CSV
START_YEAR = SAMPLE["start_year"]
MATURITY_CUTOFF = SAMPLE["maturity_cutoff_ym"]
OUTPUT_DIR = REPORT_DIR / "eda_audit"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DEV_CSV exists: {DEV_CSV.exists()} | TREASURY_CSV exists: {TREASURY_CSV.exists()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[colab_setup] 브랜치 main (fda9f461) | raw·data·outputs·report·freeze·test_hold → /content/drive/MyDrive/SNU KDT 14 4조/LendingClub | 위치 /content/kdt14th-lendingclub/notebooks
[config 2026-11-26] 공유해시 d672cc3ba054 | 전체해시 0ec794b17ddd (logit 24f0c250f0e9 | xgb 56c27c8d0e0e | test a404a78b5fa3) | STAGE=dev | 미결 3개 (B_TEST, DELTA, XGB_PARAMS), 임시값으로 실행 중
[lib_prep] 해시 e07ddd6943c5
DEV_CSV exists: True | TREASURY_CSV exists: True


---
## 1. 단일 패스 스캔 & 우측 중도절단(Right-Censoring) 컷오프 실증

원본 CSV(1.2GB, 1,755,295행)를 **딱 1번만** 청크 스캔하여 ① 스냅샷 최빈월(`May-2020`)과 만기 경계월의 미종결 비율, ② 최종 성숙 종결 표본(`2013+`, `issue_d + term ≤ 2020-04`)을 동시에 추출합니다.
- **2020-04 컷오프 근거:** `Current` 대출 최빈 결제월이 `May-2020`(약 92%)이며 1개월 결제 시차를 반영한 `2020-04` 이전 만기 코호트(36m `≤ 2017-04`, 60m `≤ 2015-04`)만 유지해야 미종결률이 **2.1% 미만**으로 통제됩니다.
- **2013+ 시작 연도 근거:** 2012년 말 LendingClub 신용조회 스키마 확장이 완료되어 `2013+` 구간이 전체 성숙 표본의 **92.3%**를 차지하면서도 구버전 정책 미충족·미수집 시점 대리변수 오염을 차단합니다.


In [2]:
chunks, cur_last_counts, censor_parts = [], [], []
raw_rows = 0
_cy = int(MATURITY_CUTOFF.split("-")[0]) if MATURITY_CUTOFF else 2020

for ch in pd.read_csv(DEV_CSV, chunksize=250_000, low_memory=False):
    raw_rows += len(ch)
    ch = drop_corrupted_rows(ch)
    issue_dt = parse_month(ch["issue_d"])
    term_m = parse_term_months(ch["term"])

    cur_last_counts.append(ch.loc[ch["loan_status"] == "Current", "last_pymnt_d"].value_counts())
    border = ((term_m == 36) & (issue_dt.dt.year == _cy - 3)) | ((term_m == 60) & (issue_dt.dt.year == _cy - 5))
    if border.any():
        b_df = pd.DataFrame({
            "term_m": term_m.loc[border].to_numpy(dtype=np.int32),
            "issue_ym": issue_dt.loc[border].dt.to_period("M").astype(str).to_numpy(),
            "resolved": ch.loc[border, "loan_status"].isin(RESOLVED_PAID | RESOLVED_DEFAULT).to_numpy(),
        })
        censor_parts.append(b_df)

    chunks.append(filter_cohort(ch, start_year=START_YEAR, maturity_cutoff_ym=MATURITY_CUTOFF, require_resolved=True))

dev_df = pd.concat(chunks, ignore_index=True)
top_cur = pd.concat(cur_last_counts).groupby(level=0).sum().sort_values(ascending=False).head(3) if cur_last_counts else pd.Series(dtype=int)
if censor_parts:
    censor_tbl = (
        pd.concat(censor_parts, ignore_index=True)
        .groupby(["term_m", "issue_ym"])["resolved"]
        .agg(n="size", unresolved_pct=lambda s: round((1.0 - s.mean()) * 100.0, 2))
        .reset_index()
    )
else:
    print("경고: 만기 경계연도 표본이 없어 표 1이 비어 있습니다.")
    censor_tbl = pd.DataFrame(columns=["term_m", "issue_ym", "n", "unresolved_pct"])

print(f"원본 {raw_rows:,}행 -> 종결·{START_YEAR}+·만기도래(≤{MATURITY_CUTOFF}) {len(dev_df):,}행")
print(f"Current 대출 최빈 last_pymnt_d: {top_cur.to_dict()}\n")
print("[표 1] 만기 경계월 미종결(우측 중도절단) 비율 (%)")
print(censor_tbl.to_string(index=False))

censor_tbl.to_csv(OUTPUT_DIR / "table_1_censoring.csv", index=False)

원본 2,340,395행 -> 종결·2013+·만기도래(≤2020-04) 1,109,665행
Current 대출 최빈 last_pymnt_d: {'May-2020': 495269, 'Jun-2020': 17570, 'Jul-2020': 4274}

[표 1] 만기 경계월 미종결(우측 중도절단) 비율 (%)
 term_m issue_ym     n  unresolved_pct
     36  2017-01 24152            0.50
     36  2017-02 20453            0.58
     36  2017-03 27805            0.76
     36  2017-04 21864            2.04
     36  2017-05 27456           10.70
     36  2017-06 27785           26.97
     36  2017-07 28738           30.23
     36  2017-08 32595           32.92
     36  2017-09 26894           34.57
     36  2017-10 26889           37.43
     36  2017-11 29656           40.36
     36  2017-12 26132           42.62
     60  2015-01 11475            0.24
     60  2015-02  7747            0.46
     60  2015-03  8486            0.46
     60  2015-04 11808            1.42
     60  2015-05 10374            7.12
     60  2015-06  9421           15.75
     60  2015-07 15096           16.87
     60  2015-08 11947           19.54
     60  

---
## 2. 만기 대응 미국 국채 매칭 & 현금흐름 복원 기반 `MIRR` 검증

- **회계 항등식 & 수수료 보정:** LendingClub 원데이터에서 $\text{total\_pymnt} \equiv \text{total\_rec\_prncp} + \text{total\_rec\_int} + \text{total\_rec\_late\_fee} + \text{recoveries}$ (오차 $\le \$0.01$)이며, `collection_recovery_fee`(부도 회수액의 약 16.8%)는 차감되어 있지 않습니다. 따라서 정기 상환 $P_i^{\text{reg}} = \max(\text{total\_pymnt}_i - \text{recoveries}_i, 0)$과 순추심 $R_i^{\text{net}} = \max(\text{recoveries}_i - \text{collection\_recovery\_fee}_i, 0)$을 분리합니다.
- **폐형식 `MIRR` 및 초과수익률:** 월별 정기 납입(최종 납입 `last_pymnt_amnt` at $t=k_i$, 이전 $k_i-1$개월 균등 배분)과 부도 후 순추심($t_{\text{rec}} = \min(k_i + 5, T_i)$)을 발행월·만기 대응 국채 수익률(**36m→`DGS3`**, **60m→`DGS5`**)의 월 실효률 $r_m = (1 + r_{f, i})^{1/12} - 1$로 계약 만기 $T_i$까지 복리 재투자합니다:
  $$\text{MIRR}_i = \left(\frac{\text{FV}_{\text{reg}} + \text{FV}_{\text{final}} + \text{FV}_{\text{rec}}}{\text{funded\_amnt}_i}\right)^{12 / T_i} - 1, \qquad R_i^{\text{excess}} = \text{MIRR}_i - r_{f, i}$$
- *(참고 비교용으로 단순 만기 연환산 `hpr_ann`도 `meta`에 병기합니다.)*


In [3]:
# 수학적 불변성 핵심 단위 검증 (전액손실=-100%, t=0 무이자 원금상환 초과수익률=0%)
_u = compute_cashflow_mirr(
    funded_amnt=np.array([10000.0, 10000.0]),
    term_m=np.array([36.0, 36.0]),
    total_pymnt=np.array([0.0, 10000.0]),
    recoveries=np.array([0.0, 0.0]),
    collection_recovery_fee=np.array([0.0, 0.0]),
    last_pymnt_amnt=np.array([0.0, 10000.0]),
    k_months=np.array([np.nan, 0.0]),
    rf_ann=np.array([0.02, 0.03]),
)
assert np.isclose(_u["mirr"][0], -1.0) and np.isclose(_u["excess_ret"][1], 0.0)

treasury_df = load_treasury(TREASURY_CSV)
meta_dev = build_meta(dev_df, treasury_df, is_labeled=True)
assert meta_dev["rf_ann"].isna().sum() == 0 and np.isfinite(meta_dev["mirr"]).all()

print(f"[PASS] MIRR 단위 검증 + {len(meta_dev):,}건 결측/Inf 0건 | 전체 부도율 = {meta_dev['y'].mean():.2%}\n")
print("[표 2-A] 상환 상태별 실현 수익률 (0=정상상환, 1=부도)")
print(meta_dev.groupby("y")[["rf_ann", "mirr", "excess_ret", "hpr_ann"]].agg(["count", "mean", "median", "std"]).round(4).to_string())

print("\n[표 2-B] 등급(Grade)별 부도율 · MIRR · 초과수익률")
print(meta_dev.groupby("grade")[["y", "rf_ann", "mirr", "excess_ret", "hpr_ann"]].mean().round(4).to_string())

print("\n[표 2-C] 만기(Term)별 부도율 · 국채수익률 · MIRR · 초과수익률")
print(meta_dev.groupby("term_m")[["y", "rf_ann", "mirr", "excess_ret", "hpr_ann"]].mean().round(4).to_string())


meta_dev.groupby("y")[["rf_ann", "mirr", "excess_ret", "hpr_ann"]].agg(["count", "mean", "median", "std"]).round(4).to_csv(OUTPUT_DIR / "table_2a_return_by_status.csv")
meta_dev.groupby("grade")[["y", "rf_ann", "mirr", "excess_ret", "hpr_ann"]].mean().round(4).to_csv(OUTPUT_DIR / "table_2b_by_grade.csv")
meta_dev.groupby("term_m")[["y", "rf_ann", "mirr", "excess_ret", "hpr_ann"]].mean().round(4).to_csv(OUTPUT_DIR / "table_2c_by_term.csv")

[PASS] MIRR 단위 검증 + 1,109,665건 결측/Inf 0건 | 전체 부도율 = 16.26%

[표 2-A] 상환 상태별 실현 수익률 (0=정상상환, 1=부도)
   rf_ann                            mirr                         excess_ret                         hpr_ann                        
    count    mean  median     std   count    mean  median     std      count    mean  median     std   count    mean  median     std
y                                                                                                                                   
0  929194  0.0105  0.0099  0.0030  929194  0.0552  0.0528  0.0223     929194  0.0447  0.0425  0.0222  929194  0.0486  0.0467  0.0230
1  180471  0.0111  0.0102  0.0032  180471 -0.1507 -0.1305  0.1437     180471 -0.1618 -0.1418  0.1433  180471 -0.1574 -0.1378  0.1432

[표 2-B] 등급(Grade)별 부도율 · MIRR · 초과수익률
            y  rf_ann    mirr  excess_ret  hpr_ann
grade                                             
A      0.0537  0.0101  0.0254      0.0153   0.0194
B      0.1171  0.0102  0.0260      0.0158   0.

---
## 3. 타깃 미사용(Target-Agnostic) 6단계 변수 선별 감사 (`141 → 37` 원천)

부도 여부($y$)와 `Test` 셋을 단 한 번도 보지 않고 `Dev`의 도메인·시점·규제·결측·비지도 공선성 규칙만으로 선별합니다:
1. **Stage 1 (비예측·자유텍스트):** `id`, `url`, `policy_code`, `title`, `emp_title`
2. **Stage 2 (시점 통제 누수 차단):** $t_3$ 사후 상환·추심(33개), $t_2$ 펀딩·발행일(4개), $t_1$ 플랫폼 심사·가격(`grade`, `sub_grade`, `int_rate`, `installment`, `verification_status*`, 6개)
3. **Stage 3 (공정대출 규제 ECOA):** `addr_state`, `zip_code` (지역 기반 Redlining 차별 방지)
4. **Stage 4 (시점 일관성 · 결측 · 준상수):** 결측 > 20% 또는 연도별 가용률 < 80%(시점 대리변수 차단) 및 최빈값 비중 > 99% 제외
5. **Stage 5 (비지도 다중공선성 제거):** 선행문헌 **Core 5Cs 16개**를 최우선 보호하면서 $|r| > 0.75$ 중복 제거 + 반복 **$\text{VIF} < 5.0$** 달성


In [4]:
def audit_variable_screening(df: pd.DataFrame, corr_th: float = 0.75, vif_th: float = 5.0) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    raw_cols = list(df.columns)
    issue_dt = parse_month(df["issue_d"])
    issue_yr = issue_dt.dt.year

    s1 = {"id", "member_id", "url", "policy_code", "desc", "title", "emp_title"} & set(raw_cols)
    s2_t1 = {"grade", "sub_grade", "int_rate", "installment", "verification_status", "verification_status_joint"} & set(raw_cols)
    s2_t2 = {"funded_amnt", "funded_amnt_inv", "initial_list_status", "issue_d"} & set(raw_cols)
    s3 = {"addr_state", "zip_code"} & set(raw_cols)
    s2_t3 = (FORBIDDEN_COLS - s1 - s2_t1 - s2_t2 - s3) & set(raw_cols)
    after_s3 = [c for c in raw_cols if c not in (s1 | s2_t1 | s2_t2 | s2_t3 | s3)]

    # Stage 4: 벡터화 결측률·연도별 가용률·준상수 필터
    miss_all = df[after_s3].isna().mean()
    min_yr_av = df[after_s3].notna().groupby(issue_yr).mean().min(axis=0)
    s4_miss, s4_const, after_s4 = [], [], []
    for c in after_s3:
        if miss_all[c] > 0.20 or min_yr_av[c] < 0.80:
            s4_miss.append(c)
        elif df[c].value_counts(normalize=True, dropna=True).iloc[0] > 0.99:
            s4_const.append(c)
        else:
            after_s4.append(c)

    # Stage 5: 행 독립 수치 정제 후 비지도 |r|>corr_th + 우선순위 보존 VIF<vif_th
    cat_cols = [c for c in ("home_ownership", "purpose") if c in after_s4]
    money_cols = {k for k, v in FEATURE_SPEC.items() if v[1].startswith("log1p")} | {"tot_hi_cred_lim", "total_rev_hi_lim", "avg_cur_bal"}
    num_map, feat_to_raw = {}, {}
    for c in [x for x in after_s4 if x not in cat_cols]:
        if c == "term":
            name, arr = "term_60m", (parse_term_months(df[c]) == 60).to_numpy(dtype=np.float64)
        elif c == "earliest_cr_line":
            name, arr = "credit_hist_len", months_between(issue_dt, parse_month(df[c])).clip(lower=0).to_numpy(dtype=np.float64)
        elif c == "emp_length":
            s = df[c].astype("string").str.strip().str.lower()
            v = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce").mask(s.str.startswith("<", na=False), 0.0).clip(0, 10)
            name, arr = "emp_length", v.to_numpy(dtype=np.float64)
        elif c == "revol_util":
            name, arr = "revol_util", pd.to_numeric(df[c].astype("string").str.strip().str.rstrip("%"), errors="coerce").clip(0, 150).to_numpy(dtype=np.float64)
        elif c == "dti":
            d = pd.to_numeric(df[c], errors="coerce").where(lambda x: (x >= 0) & (x < 999))
            name, arr = "dti", d.clip(upper=d.quantile(PREP["dti_clip_q"])).to_numpy(dtype=np.float64)
        elif c in money_cols:
            v = pd.to_numeric(df[c], errors="coerce")
            v = v.where(v > 0, np.nan) if c == "annual_inc" else v.clip(lower=0)
            name, arr = f"log_{c}", np.log1p(v).to_numpy(dtype=np.float64)
        else:
            name, arr = c, pd.to_numeric(df[c], errors="coerce").clip(lower=0).to_numpy(dtype=np.float64)
        num_map[name], feat_to_raw[name] = arr, c

    num_df = pd.DataFrame(num_map)
    miss_r = num_df.isna().mean().to_dict()
    num_imp = num_df.apply(lambda s: s.fillna(s.max() if s.name == "mths_since_recent_inq" else s.median()))
    corr_abs = num_imp.corr().abs()

    core_order = [v[0] for v in get_feature_spec("core_t0").values() if v[1] != "cat"]
    core_set = set(core_order)
    order = sorted(num_imp.columns, key=lambda c: (0, core_order.index(c), 0.0, c) if c in core_set else (1, 0, miss_r[c], c))

    kept, s5_corr = [], []
    for c in order:
        if any(corr_abs.loc[c, k] > corr_th for k in kept):
            s5_corr.append(feat_to_raw[c])
        else:
            kept.append(c)

    X_d = num_imp[kept].copy()
    dummy_cols = []
    for cat in cat_cols:
        s = df[cat].astype("string").str.strip().str.upper().fillna("OTHER")
        known = set(s.value_counts(normalize=True)[lambda v: v >= PREP["rare_cat_min_freq"]].index)
        s = s.where(s.isin(known), "OTHER")
        for lv in [x for x in s.value_counts().index if x != CAT_REFERENCE[cat]]:
            dcol = f"{cat}_{lv}"
            X_d[dcol] = (s == lv).to_numpy(dtype=np.float64)
            dummy_cols.append(dcol)

    protected = core_set | set(dummy_cols)
    s5_vif = []
    while (vifs := compute_vif(X_d)).max() >= vif_th:
        non_prot = vifs.drop(index=[c for c in protected if c in vifs.index])
        victim = str(non_prot.idxmax()) if (len(non_prot) > 0 and non_prot.max() >= 3.0) else str(vifs.idxmax())
        s5_vif.append(feat_to_raw.get(victim, victim))
        X_d = X_d.drop(columns=[victim])

    final_vifs = compute_vif(X_d)
    final_raw = [feat_to_raw[c] for c in X_d.columns if c not in dummy_cols] + cat_cols
    n = len(raw_cols)
    funnel = pd.DataFrame([
        ("0. 원본 LendingClub 칼럼", 0, n),
        ("1. 식별자·상수·자유텍스트 제외", -len(s1), n := n - len(s1)),
        ("2-a. t3 사후 상환·추심·갱신점수 제외", -len(s2_t3), n := n - len(s2_t3)),
        ("2-b. t2 펀딩·발행일 제외", -len(s2_t2), n := n - len(s2_t2)),
        ("2-c. t1 플랫폼 심사·가격 책정 제외", -len(s2_t1), n := n - len(s2_t1)),
        ("3. 공정대출 규제 필터 (ECOA)", -len(s3), n := n - len(s3)),
        ("4-a. 시점 불일치·고결측(>20%/연도가용<80%)", -len(s4_miss), n := n - len(s4_miss)),
        ("4-b. 준상수 제외 (최빈값 >99%)", -len(s4_const), n := n - len(s4_const)),
        (f"5-a. 비지도 상관 중복 제거 (|r|>{corr_th})", -len(s5_corr), n := n - len(s5_corr)),
        (f"5-b. 비지도 다중공선성 제거 (VIF>={vif_th})", -len(s5_vif), n := n - len(s5_vif)),
    ], columns=["단계", "제외 수", "잔여 원천 칼럼"])

    spec_tbl = pd.DataFrame([
        {
            "feature": c,
            "group": "Core 5Cs" if c in core_set else ("Dummy" if c in dummy_cols else "Extended t0"),
            "miss_pct": round(miss_r.get(c, 0.0) * 100.0, 2),
            "VIF": round(float(final_vifs[c]), 2),
            "monotone": next((v[3] for v in FEATURE_SPEC.values() if v[0] == c), 0),
        }
        for c in X_d.columns
    ])
    return funnel, spec_tbl, final_raw

funnel_tbl, spec_tbl, screened_raw = audit_variable_screening(dev_df)
if STAGE == "final":
    assert set(screened_raw) == set(FEATURE_SPEC), "감사 결과와 lib_prep.FEATURE_SPEC이 일치해야 합니다!"
elif set(screened_raw) != set(FEATURE_SPEC):
    print("경고(STAGE != 'final'): 감사 선별 결과와 FEATURE_SPEC 차이:",
          "추가", sorted(set(screened_raw) - set(FEATURE_SPEC)), "누락", sorted(set(FEATURE_SPEC) - set(screened_raw)))

print("[표 3-A] 141개 원본 칼럼 -> 6단계 변수 선별 깔때기")
print(funnel_tbl.to_string(index=False))
print(f"\n[표 3-B] Stage 5 감사 통과 원천 피처 ({len(screened_raw)}개 원천 -> 감사기준 {len(spec_tbl)}열, 최대 VIF={spec_tbl['VIF'].max():.2f} < 5.0)")
print(spec_tbl.to_string(index=False))


funnel_tbl.to_csv(OUTPUT_DIR / "table_3a_screening_funnel.csv", index=False)
spec_tbl.to_csv(OUTPUT_DIR / "table_3b_feature_spec.csv", index=False)
print("저장:", sorted(p.name for p in OUTPUT_DIR.glob("*.csv")))

[표 3-A] 141개 원본 칼럼 -> 6단계 변수 선별 깔때기
                            단계  제외 수  잔여 원천 칼럼
          0. 원본 LendingClub 칼럼     0       141
            1. 식별자·상수·자유텍스트 제외    -5       136
      2-a. t3 사후 상환·추심·갱신점수 제외   -33       103
             2-b. t2 펀딩·발행일 제외    -4        99
       2-c. t1 플랫폼 심사·가격 책정 제외    -6        93
          3. 공정대출 규제 필터 (ECOA)    -2        91
4-a. 시점 불일치·고결측(>20%/연도가용<80%)   -33        58
        4-b. 준상수 제외 (최빈값 >99%)    -6        52
  5-a. 비지도 상관 중복 제거 (|r|>0.75)   -12        40
  5-b. 비지도 다중공선성 제거 (VIF>=5.0)    -3        37

[표 3-B] Stage 5 감사 통과 원천 피처 (37개 원천 -> 감사기준 47열, 최대 VIF=4.44 < 5.0)
                       feature       group  miss_pct  VIF  monotone
                fico_range_low    Core 5Cs      0.00 2.26        -1
               credit_hist_len    Core 5Cs      0.00 1.43        -1
                   delinq_2yrs    Core 5Cs      0.00 2.31         1
                inq_last_6mths    Core 5Cs      0.00 1.50         1
                       pub_rec    Core